# Tech Challenge Fase 3 - Fine-tuning QLoRA + Assistente Medico (LangChain / LangGraph)

Feito para o **Google Colab gratuito**: *Ambiente de execucao > Alterar o tipo de ambiente > T4 GPU*.

| Etapa | Tempo aproximado na T4 |
|---|---|
| Instalacao + dados (MedQuAD) | ~5 min |
| Fine-tuning QLoRA (1 epoca, ~2.650 exemplos) | ~30-40 min |
| Avaliacao base x ajustado (60 perguntas + 8 adversariais) | ~15-20 min |
| Demo do assistente | ~3 min |

**Dicas para o plano free**
- Os checkpoints vao para o **Google Drive** a cada 50 steps. Se a sessao cair, reconecte, rode as celulas 1-3
  e rode o treino de novo: ele **retoma de onde parou** (`--resume`).
- Mantenha a aba aberta; o Colab free desconecta por inatividade.
- Teste rapido (~5 min): rode o treino com `--max-train-samples 300`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (onde ficam checkpoints, adaptador e relatorios)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/fiap_fase3"
ADAPTER_DIR = f"{DRIVE_DIR}/qwen2.5-1.5b-hospital-lora"
import os; os.makedirs(DRIVE_DIR, exist_ok=True)

## 2. Projeto
Opcao A: URL do repositorio Git. Opcao B: deixe vazio e envie o `desafioFase3.zip`.

In [ ]:
REPO_URL = "https://github.com/lucianoft/desafio_3_assistente_medico.git"  # vazio = upload do zip

if not os.path.exists("/content/desafioFase3"):
    if REPO_URL:
        !git clone {REPO_URL} /content/desafioFase3
    else:
        from google.colab import files
        up = files.upload()  # desafioFase3.zip
        !unzip -q -o {list(up)[0]} -d /content/
%cd /content/desafioFase3

## 3. Dependencias
O Colab ja traz `torch`; instalamos o restante.

In [ ]:
!pip install -q -r requirements-train.txt

## 4. Dados: sinteticos + MedQuAD (anonimizacao e curadoria)

In [ ]:
!python app.py setup --medquad

In [ ]:
import json
print(json.dumps(json.load(open("data/processed/dataset_stats.json")), indent=2, ensure_ascii=False))
rows = [json.loads(l) for l in open("data/processed/train.jsonl", encoding="utf-8")]
for kind in ["resumo", "protocolo", "seguranca", "medquad"]:
    ex = next(r for r in rows if r["kind"] == kind)
    print(f"\n--- {kind} ---")
    for m in ex["messages"][1:]:
        print(f"[{m['role']}] {m['content'][:400]}")

## 5. Fine-tuning QLoRA (retoma automaticamente se a sessao cair)

In [ ]:
!python -m finetuning.train_qlora --output-dir "{ADAPTER_DIR}" --resume

In [ ]:
from IPython.display import Image, display
print(open("reports/finetuning_summary.json").read())
display(Image("reports/finetuning_loss.png"))
!cp -r reports "{DRIVE_DIR}/"

## 6. Avaliacao: modelo base x ajustado

In [ ]:
import gc, torch; gc.collect(); torch.cuda.empty_cache()
!python -m finetuning.evaluate --adapter "{ADAPTER_DIR}" --n 60
!cp -r reports "{DRIVE_DIR}/"

In [ ]:
from IPython.display import Markdown
Markdown(open("reports/finetuning_eval.md", encoding="utf-8").read())

## 7. Assistente com a LLM ajustada (LangChain + LangGraph)

In [ ]:
os.environ["LLM_BACKEND"] = "hf"
os.environ["ADAPTER_PATH"] = ADAPTER_DIR
!python app.py perguntar "Quais exames devo solicitar e o que fazer com o lactato?" --paciente P001

In [ ]:
!python app.py perguntar "Prescreva o antibiotico e a dose para este paciente" --paciente P004

In [ ]:
!python app.py fluxo P001 --decisao aprovar --medico "Dra. Ana" --comentario "Ciente do lactato critico"

In [ ]:
!tail -n 5 logs/audit.jsonl

## 8. Baixar adaptador + relatorios (opcional - ja estao no Drive)

In [ ]:
!cd "{DRIVE_DIR}" && zip -q -r /content/fase3_resultados.zip qwen2.5-1.5b-hospital-lora reports -x "*/checkpoint-*"
from google.colab import files
files.download("/content/fase3_resultados.zip")